In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

prototype_NRB = pd.read_csv("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/26-01-2026.csv")

df = prototype_NRB.copy()

r = df["range_m"].to_numpy(float)

# all time columns (everything except range_m)
time_cols = [c for c in df.columns if c != "range_m"]

# NRB as 2D matrix: shape (n_range, n_time)
NRB = df[time_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)

print("r:", r.shape, "NRB:", NRB.shape)  # (n_range,) (n_range, n_time)

r: (4000,) NRB: (4000, 48)


In [3]:
def time_str_to_minutes(t: str) -> int:
    hh, mm = t.split(".")
    return int(hh) * 60 + int(mm)

t_min = np.array([time_str_to_minutes(c) for c in time_cols], dtype=int)

In [5]:
df

,range_m,00.05,00.35,01.05,01.35,02.05,02.35,03.05,03.35,04.05,...,19.05,19.35,20.05,20.35,21.05,21.35,22.05,22.35,23.05,23.35
0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,3.75,0.000050,0.000071,0.000047,0.000054,0.000043,0.000051,0.000044,0.000146,0.000112,...,0.000141,0.000127,0.000119,0.000048,0.000042,0.000142,0.000039,0.000144,0.000074,0.000113
2,7.50,0.000312,0.000553,0.000281,0.000340,0.000241,0.000309,0.000253,0.001596,0.001066,...,0.001823,0.001493,0.001301,0.000287,0.000247,0.001711,0.000219,0.001714,0.000576,0.001180
3,11.25,0.000968,0.001848,0.000840,0.001070,0.000679,0.000933,0.000712,0.004862,0.003307,...,0.006501,0.005209,0.004489,0.000887,0.000732,0.005838,0.000625,0.005815,0.001896,0.003833
4,15.00,0.001721,0.003286,0.001494,0.001903,0.001207,0.001658,0.001265,0.008631,0.005869,...,0.011549,0.009253,0.007974,0.001577,0.001302,0.010368,0.001111,0.010327,0.003369,0.006807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,14981.25,-1.083239,-0.883833,-1.054446,-1.150221,-1.362754,-1.019934,-1.296968,-0.761901,-0.926122,...,-0.844521,-1.002768,-1.054841,-1.277765,-1.123081,-1.236424,-1.236499,-1.028746,-1.033038,-0.945965
3996,14985.00,-0.958644,-0.884276,-0.994148,-1.150797,-1.250649,-1.020445,-1.297617,-1.287270,-1.202316,...,-1.131133,-1.299428,-1.250087,-1.278405,-1.233627,-1.237043,-1.287770,-1.029261,-1.115139,-1.043481
3997,14988.75,-1.021724,-0.884718,-0.994646,-1.085044,-1.420540,-1.084408,-1.412537,-0.972764,-1.202918,...,-0.940811,-1.102541,-1.153305,-1.401534,-0.904128,-1.131659,-1.339091,-1.244665,-1.197322,-1.141095
3998,14992.50,-1.210129,-1.039026,-1.116917,-1.151949,-1.251901,-1.021467,-1.184590,-1.183456,-1.019516,...,-0.750297,-1.004275,-1.348795,-1.157135,-1.179815,-0.920113,-1.136953,-1.137790,-1.197921,-1.044526


In [ ]:
def haar_covariance(beta, z, a):
    dz = np.mean(np.diff(z))
    half = int((a / 2) / dz)

    W = np.full_like(beta, np.nan, dtype=float)

    for i in range(half, len(beta) - half):
        lower = beta[i - half:i]
        upper = beta[i:i + half]
        W[i] = np.mean(lower) - np.mean(upper)

    return W

def find_transition_zone(z, W, frac_low=0.3, frac_high=0.7):
    Wmax = np.nanmax(W)
    imax = np.nanargmax(W)

    # search downward → H1
    for i in range(imax, 0, -1):
        if W[i] < frac_low * Wmax:
            H1 = z[i]
            break

    # search upward → H2
    for i in range(imax, len(W)):
        if W[i] < frac_high * Wmax:
            H2 = z[i]
            break

    return H1, H2
    
# choose dilation ~ expected inversion thickness
a = 120.0  # meters (Brooks used 40–200 m typically)
z = df["range_m"]
beta = 
W = haar_covariance(beta, z, a)

H1, H2 = find_transition_zone(z, W)

BL_top = H1   # physically meaningful BL height
